# The bench panel: size and spend to date

Every run the bench store holds, and every model call those runs made, priced against the price snapshot the call ran
under (`docs/dev/e2e-harness.md`; `tests/e2e/bench/spend.mjs` is the same sum for one checkout's scores). A call is
priced by the bench's own code, `price-calls.mjs` beside this notebook, so the join from a model to its price is the
one the scoreboard uses, not a second copy of it.

In [1]:
import json, subprocess, tempfile
from pathlib import Path
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "bench"
DATA = HERE / "data" / "spend-to-date"
pd.set_option("display.width", 160)

_con = None
def con():
    """The store, opened only when an extract has to read it (a pinned run never connects)."""
    global _con
    _con = _con or wmlnb.store()
    return _con

## The rows, pinned

The store is at-least-once: overlapping pushes put a row in several objects. Its views keep one row per key: a run per
`run`, a call per (`run`, `call`), a price snapshot per `hash`. The runs are kept whole; the calls are kept as their
priced result only (the snapshot bodies are megabytes each, and only their prices matter here).

In [2]:
RUNS = '''
SELECT run, sweep, model, task, local::BOOLEAN AS local, scored, passed, "at" FROM scores ORDER BY "at"'''
CALLS = '''SELECT run, call, kind, model, usage FROM calls'''
SNAPSHOTS = '''SELECT hash, body FROM price_snapshots'''

def priced_calls():
    # Rows as plain dicts, not a DataFrame's: pandas turns a missing model into NaN, which is not JSON.
    rel = con().sql(CALLS)
    calls = [dict(zip(rel.columns, r)) for r in rel.fetchall()]
    drivers = dict(con().sql("SELECT run, model FROM scores").fetchall())
    snapshots = dict(con().sql(SNAPSHOTS).fetchall())
    with tempfile.TemporaryDirectory() as d:
        src, out = Path(d) / "in.json", Path(d) / "out.json"
        src.write_text(json.dumps({"calls": calls, "drivers": drivers, "snapshots": snapshots}, default=int))
        subprocess.run(["npx", "tsx", str(HERE / "price-calls.mjs"), str(src), str(out)], cwd=wmlnb.ROOT, check=True)
        return json.loads(out.read_text())

runs = pd.DataFrame(wmlnb.extract(DATA / "runs.json", lambda: con().sql(RUNS).df().astype({"at": str}).to_dict("records"),
                                  source={"store": "bench store (sync.mjs)", "view": "scores", "query": RUNS.strip()}))
calls = pd.DataFrame(wmlnb.extract(DATA / "priced-calls.json", priced_calls,
                                   source={"store": "bench store (sync.mjs)", "views": ["calls", "scores", "price_snapshots"],
                                           "query": CALLS, "priced_by": "notebooks/bench/price-calls.mjs (tests/e2e/bench/cost.mjs)"}))
pd.DataFrame(wmlnb.provenance(DATA / "runs.json", DATA / "priced-calls.json"))

,extract,sha256,rows,extracted_at,commit,query,store,view,priced_by,views
0,notebooks/bench/data/spend-to-date/runs.json,7d029d928ce278fd,2663,2026-10-21T17:34:04+00:00,de45e819f2bd,"SELECT run, sweep, model, task, local::BOOLEAN...",bench store (sync.mjs),scores,NaN,NaN
1,notebooks/bench/data/spend-to-date/priced-call...,e9ce89f21d14bdad,14619,2026-10-21T17:34:06+00:00,de45e819f2bd,"SELECT run, call, kind, model, usage FROM calls",bench store (sync.mjs),NaN,notebooks/bench/price-calls.mjs (tests/e2e/ben...,"[calls, scores, price_snapshots]"


## The panel

In [3]:
pd.Series({"runs": len(runs), "models": runs.model.nunique(), "local models": runs[runs.local == True].model.nunique(),
           "tasks": runs.task.nunique(), "sweeps": runs.sweep.nunique(), "first": runs["at"].min()[:10], "last": runs["at"].max()[:10]})

runs                  2663
models                  21
local models             9
tasks                   23
sweeps                  26
first           2026-10-09
last            2026-10-21
dtype: object

## Spend

`computed` is the call's tokens at its snapshot's prices; `reported` is what the provider said the call cost, where it
said so. A call on a local model costs electricity, not tokens, and is counted apart. Each call's model is its run's
driver when the call does not name one, as the scoreboard reads it.

In [4]:
calls["driver"] = calls.run.map(dict(zip(runs.run, runs.model)))
calls["state"] = calls.apply(lambda c: "local" if c.local else "priced" if pd.notna(c.computed) else "reported only" if pd.notna(c.reported) else "unpriced", axis=1)
total = pd.Series({"calls": len(calls), "computed USD": round(calls.computed.sum(), 2), "priced calls": int(calls.computed.notna().sum()),
                   "reported USD": round(calls.reported.sum(), 2), "reported calls": int(calls.reported.notna().sum()),
                   "local calls": int(calls.local.sum()), "unpriced calls": int((calls.state == "unpriced").sum())})
total

calls             14619.00
computed USD          7.01
priced calls       3378.00
reported USD          1.74
reported calls     1723.00
local calls        5527.00
unpriced calls     5054.00
dtype: float64

In [5]:
(calls[calls.computed.notna()].groupby("driver").agg(calls=("call", "size"), computed_usd=("computed", "sum"))
 .round({"computed_usd": 2}).sort_values("computed_usd", ascending=False))

,calls,computed_usd
driver,,
litellm.google/gemini-flash-latest,553,4.20
openrouter.minimax/minimax-m3,973,1.32
deepseek.deepseek-flash,859,0.65
openrouter.xiaomi/mimo-v2.6-pro,554,0.32
litellm.anthropic/claude-opus-5-5,11,0.17
openrouter.openai/gpt-6-luna,381,0.11
moonshot.kimi-k3,12,0.10
deepseek.deepseek-v4-pro,23,0.08
litellm.anthropic/claude-sonnet-5-5,12,0.07


## The undercount

A cloud call with no price snapshot is in neither sum. Most of them ran before the bench set a price service at all; the
rest are the first calls of a cell. Each cell starts in a fresh browser profile with no snapshot, a call waits 15 s for
the first one and then runs without it, and the refresh took longer than that: on panel-policy-sh the first two of every
cloud cell's four calls went unpriced. #950 (2026-10-21) has the bench fetch the snapshot before a cell's first turn, so
sweeps from then on price every call; the sweeps before it stay undercounted.

In [6]:
un = calls[calls.state == "unpriced"]
print(f"{len(un)} unpriced calls ({len(un) / len(calls):.0%} of all calls), by reason:")
display(un.why.value_counts().to_frame("calls"))
sweep = calls.run.map(dict(zip(runs.run, runs.sweep)))
(pd.DataFrame({"sweep": sweep, "unpriced": calls.state == "unpriced", "priced": calls.state == "priced"})
 .groupby("sweep").sum().query("unpriced > 0 and priced > 0").sort_values("unpriced", ascending=False).head(15))

5054 unpriced calls (35% of all calls), by reason:


,calls
why,
no price snapshot recorded,5054


,unpriced,priced
sweep,,
panel-policy-fetch,33,66
panel-policy-sh,17,18


The table lists the sweeps that have priced AND unpriced calls: the ones where the fresh-profile bug cost a cell its
first calls' prices, rather than sweeps that ran with no price service.

Until 2026-10-21, 3,873 calls of the main clone's scoreboard were in no store object (`prompt-budget-tool-details`,
`tool-details-v2` and others): `merge-when-green --keep-bench` had made its `calls` table without its key, every id was
NULL, and a push sends only ids past its watermark. The table was rebuilt with its keys and pushed (#981 fixed
`keepBench`); every call in a checkout's scoreboard is now in the store.